In [12]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import numpy as np
# import matplotlib.pyplot as plt 
# import threading
# from math import modf

# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [10]:
def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H1, 0, 4500)
    rates_frame = pd.DataFrame(rates)

    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame['rsi'] = RSI(rates_frame['close'], 24)
#     rates_frame['slope'] = go(rates_frame)
#     rates_frame['sma']= rates_frame['close'].rolling(window=200).mean()

    return rates_frame

In [42]:
symbol = "GBPUSD"
a= get_values(symbol)
a

,open,high,low,close,tick_volume,spread,real_volume,rsi
time,,,,,,,,
2021-01-18 11:00:00,1.35272,1.35395,1.35195,1.35286,4170,9,0,NaN
2021-01-18 12:00:00,1.35287,1.35475,1.35255,1.35397,3212,9,0,NaN
2021-01-18 13:00:00,1.35400,1.35558,1.35345,1.35480,3510,9,0,NaN
2021-01-18 14:00:00,1.35485,1.35546,1.35306,1.35338,2978,9,0,NaN
2021-01-18 15:00:00,1.35338,1.35572,1.35294,1.35551,2881,9,0,NaN
...,...,...,...,...,...,...,...,...
2021-10-06 18:00:00,1.35677,1.35724,1.35497,1.35660,7136,9,0,42.755161
2021-10-06 19:00:00,1.35659,1.35730,1.35532,1.35540,4393,9,0,39.930718
2021-10-06 20:00:00,1.35541,1.35766,1.35533,1.35726,3343,9,0,45.729335


In [28]:
def go(a):
    g = []
    for i in range(0, len(a)):
        g.append(slope(0, a.iloc[i-5].rsi, 5, a.iloc[i].rsi))
    return g

In [29]:
def slope(x1, y1, x2, y2):
    return (y2-y1)/(x2-x1)

In [30]:
def RSI(series, period):
    delta = series.diff().dropna()
    u = delta * 0
    d = u.copy()
    u[delta > 0] = delta[delta > 0]
    d[delta < 0] = -delta[delta < 0]
    u[u.index[period-1]] = np.mean( u[:period] ) #first value is sum of avg gains
    u = u.drop(u.index[:(period-1)])
    d[d.index[period-1]] = np.mean( d[:period] ) #first value is sum of avg losses
    d = d.drop(d.index[:(period-1)])
    rs = pd.DataFrame.ewm(u, com=period-1, adjust=False).mean() / \
         pd.DataFrame.ewm(d, com=period-1, adjust=False).mean()
    return 100 - 100 / (1 + rs)

In [31]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [32]:
for i in range(43, len(a)):
    print(a.iloc[i].name)
    break

2021-01-20 06:00:00


In [46]:
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
b = a
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 2.0
p= []
k = 0
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 1
global_loss = []
conti = []
lot = 0.02
low_buy = 0.0
low_sell = 0.0
checks1 = 0
m = ""

for i in range(200, len(a)):
        if a.iloc[i].name.hour == 0 and a.iloc[i-24].open > a.iloc[i-1].close and a.iloc[i].rsi >= 38.0 and check == 0:
            if a.iloc[i].rsi > 40.0:
                m = 1
            buy_price = a.iloc[i].open
            print("#"*20)
            print(a.iloc[i].name, "buy")
            
            print("*"*20)

            check = 1 
            checks = 0
            up = 0


        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            if m == 1:
                mul = 2

            if pp>=4.0 and check == 1:
                profit.append(pp*mul)
                print(f"{pp*mul}--  {a.iloc[i].name}___ {a.iloc[i].rsi} {mul} buy")
                check = 0
                mul = 1

            if a.iloc[i].name.hour == 20 and check == 1:
                profit.append(pp*mul)
                print(f"{pp*mul}--  {a.iloc[i].name}___ {a.iloc[i].rsi} {mul} buyy")
                check = 0
                mul = 1
            if a.iloc[i].name.hour == 16 and pp < -1.0 and check == 1:
                profit.append(pp*mul)
                print(f"{pp*mul}--  {a.iloc[i].name}___ {a.iloc[i].rsi} {mul} buzz")
                check = 0
                mul = 1
            if a.iloc[i].rsi < 31.0 and check == 1:
                profit.append(pp*mul)
                print(f"{pp*mul}--  {a.iloc[i].name}___ {a.iloc[i].rsi} {mul} buzz")
                check = 0
                mul = 1
            m = 0



####################
2021-02-01 00:00:00 buy
********************
8.88--  2021-02-01 03:00:00___ 53.40937087687018 2 buy
####################
2021-02-02 00:00:00 buy
********************
13.56--  2021-02-02 03:00:00___ 49.39774937998561 2 buy
####################
2021-02-04 00:00:00 buy
********************
9.16--  2021-02-04 14:00:00___ 55.02622644795948 2 buy
####################
2021-02-12 00:00:00 buy
********************
14.64--  2021-02-12 17:00:00___ 59.526443617446446 2 buy
####################
2021-02-17 00:00:00 buy
********************
-19.44--  2021-02-17 16:00:00___ 42.54513828887521 2 buzz
####################
2021-02-18 00:00:00 buy
********************
21.8--  2021-02-18 10:00:00___ 59.20146482549624 2 buy
####################
2021-03-01 00:00:00 buy
********************
17.56--  2021-03-01 01:00:00___ 46.56247005917067 2 buy
####################
2021-03-02 00:00:00 buy
********************
9.48--  2021-03-02 17:00:00___ 52.27111653703666 2 buy
####################
2021

-7.4399999999999995--  2021-09-17 16:00:00___ 40.74591024613301 2 buzz
####################
2021-09-23 00:00:00 buy
********************
4.86--  2021-09-23 06:00:00___ 48.07520576330513 1 buy
####################
2021-09-27 00:00:00 buy
********************
14.92--  2021-09-27 11:00:00___ 55.0314126429141 2 buy


In [47]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))

print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

385.2
Total negative sm -->-224.53999999999994
Total negative -->17
Total positive sm -->609.74
Total positive -->57
Length 74


In [45]:
for i in profit:
    if i<0.0:
        print(i)

-19.44
-37.16
-11.84
-4.36
-4.16
-29.36
-16.8
-31.32
-3.8
-9.92
-30.4
-6.82
-7.64
-2.76
-11.36
-18.2
-7.12
-2.92
-16.76
-11.48
-3.88
-8.24
-3.2800000000000002
-33.96
-12.8
-3.94
-2.04
-7.4399999999999995


In [ ]:
#Works with GBPUSD
#0.05
b = a
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 2.0
p= []
k = 0
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
lot = 0.05
low_buy = 0.0
low_sell = 0.0
checks1 = 0
m = ""

for i in range(20, len(a)):
        if a.iloc[i].name.hour == 0 and a.iloc[i-24].open > a.iloc[i-1].close and a.iloc[i].rsi > 40.0 and check == 0:
            buy_price = a.iloc[i].open
            print("#"*20)
            print(a.iloc[i].name, "buy")
            
            print("*"*20)

            check = 1 
            checks = 0
            up = 0

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            if pp>=10.0:
                profit.append(pp)
                print(f"{pp}--  {a.iloc[i].name}___ {a.iloc[i].rsi} buy")
                check = 0
            if a.iloc[i].name.hour == 20:
                profit.append(pp)
                print(f"{pp}--  {a.iloc[i].name}___ {a.iloc[i].rsi}  buyy")
                check = 0
            if a.iloc[i].name.hour == 16 and pp < -1.0:
                profit.append(pp)
                print(f"{pp}--  {a.iloc[i].name}___ {a.iloc[i].rsi}  buyy")
                check = 0

